## Config

In [ ]:
import os

# ── Adapter source (your uploaded rank-64 adapter dataset) ─────────────────
SRC_ADAPTER_DIR = "/kaggle/input/my-rank64-adapter"

# ── LoRA training config (must match what was used during training) ─────────
LORA_RANK       = 64
LORA_ALPHA      = 128
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print(f"src_dir : {SRC_ADAPTER_DIR}")
print(f"rank    : {LORA_RANK}  alpha: {LORA_ALPHA}")
print(f"base    : {BASE_MODEL_NAME}")
print(f"Files   : {os.listdir(SRC_ADAPTER_DIR)}")


## SVD Compress rank 64 → 32

In [ ]:
# ── SVD Compress rank 64 → 32 ───────────────────────────────────────────────
# Reference: https://arxiv.org/pdf/2602.10993 (Algorithm 2)
import torch, json, os, gc
from safetensors.torch import load_file, save_file

SVD_TARGET_RANK = 32
SVD_ALPHA_RATIO = LORA_ALPHA / LORA_RANK   # 128 / 64 = 2.0
SVD_NEW_ALPHA   = int(SVD_TARGET_RANK * SVD_ALPHA_RATIO)  # 64

src_dir = SRC_ADAPTER_DIR
out_dir = "/kaggle/working/sft_adapter_svd"
os.makedirs(out_dir, exist_ok=True)

# Use GPU if available, else CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"SVD compress: rank {LORA_RANK} → {SVD_TARGET_RANK}, alpha {LORA_ALPHA} → {SVD_NEW_ALPHA}")
print(f"Device: {device}")

weights = load_file(os.path.join(src_dir, "adapter_model.safetensors"))
print(f"Loaded {len(weights)} tensors")

compressed = {}
processed_b = set()
n = 0
for key, tensor in weights.items():
    if "lora_A" in key:
        key_b = key.replace("lora_A", "lora_B")
        if key_b not in weights:
            compressed[key] = tensor
            continue
        # Move to device, compute, move back to CPU immediately
        A = tensor.float().to(device)
        B = weights[key_b].float().to(device)
        W = B @ A
        del A, B
        U, S, Vh = torch.linalg.svd(W, full_matrices=False)
        del W
        k = SVD_TARGET_RANK
        sqrt_S = torch.sqrt(S[:k])
        new_A = (sqrt_S.unsqueeze(1) * Vh[:k, :]).to(tensor.dtype).contiguous().cpu()
        new_B = (U[:, :k] * sqrt_S.unsqueeze(0)).to(tensor.dtype).contiguous().cpu()
        del U, S, Vh, sqrt_S
        compressed[key]  = new_A
        compressed[key_b] = new_B
        processed_b.add(key_b)
        n += 1
        if n % 100 == 0:
            print(f"  Processed {n} lora pairs...", flush=True)
            if device.type == "cuda":
                torch.cuda.empty_cache()
            gc.collect()
    elif "lora_B" in key and key not in processed_b:
        compressed[key] = tensor
    elif "lora_A" not in key and "lora_B" not in key:
        compressed[key] = tensor

print(f"SVD done. {n} lora pairs compressed.")

# Ensure all tensors contiguous before saving
compressed = {k: v.contiguous() for k, v in compressed.items()}
save_file(compressed, os.path.join(out_dir, "adapter_model.safetensors"))
print("Compressed weights saved.")

# Update adapter_config.json
with open(os.path.join(src_dir, "adapter_config.json")) as f:
    cfg = json.load(f)
cfg["r"]          = SVD_TARGET_RANK
cfg["lora_alpha"] = SVD_NEW_ALPHA
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(os.path.join(out_dir, "adapter_config.json"), "w") as f:
    json.dump(cfg, f, indent=2)

orig_mb = os.path.getsize(os.path.join(src_dir, "adapter_model.safetensors")) / 1024**2
comp_mb = os.path.getsize(os.path.join(out_dir, "adapter_model.safetensors")) / 1024**2
print(f"Original:   {orig_mb:.1f} MB  (rank {LORA_RANK})")
print(f"Compressed: {comp_mb:.1f} MB  (rank {SVD_TARGET_RANK})")
print(f"adapter_config.json → r={SVD_TARGET_RANK}, lora_alpha={SVD_NEW_ALPHA}")
print("SVD compression done.")


## Create submission.zip

In [ ]:
import json, os, shutil, zipfile

OUTPUT_DIR             = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

src_adapter_dir = "/kaggle/working/sft_adapter_svd"  # SVD-compressed adapter
required_files  = ["adapter_config.json", "adapter_model.safetensors"]

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

# Set inference config
config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path) as f:
    cfg = json.load(f)
cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"] = True
cfg["lora_dropout"]   = 0.0
with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

print(f"\nsubmission.zip: {os.path.getsize(zip_path)/1024/1024:.1f} MB")
print("Done! Ready to submit.")
